# 01 — Coleta de dados

Este notebook coleta as séries dos sete ativos definidos em `config.yaml`, os
indicadores do BCB-SGS e do FRED e as medidas de volatilidade implícita. As saídas
são salvas em `data/raw/`, junto com os relatórios de cobertura.

A coleta do Yahoo mantém as referências de futuros para comparação. No experimento
principal, soja e café usam as séries reconstruídas por
`scripts/build_public_futures.py`. Os indicadores físicos do CEPEA ficam disponíveis
para uma análise separada.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.collectors import (
    collect_bcb_sgs,
    collect_cepea_spot,
    collect_equities_fx,
    collect_fred,
    collect_futures_raw,
    collect_implied_vol_us,
    load_bloomberg_implied_vol_br,
    adjust_ohlc_roll,
)
from src.data.validation import check_coverage, summarize_checks

config = yaml.safe_load((PROJECT_ROOT / "config.yaml").read_text(encoding="utf-8"))
start, end = config["sample"]["start_date"], config["sample"]["end_date"]

OUT_DIR = PROJECT_ROOT / "data" / "raw"
OUT_DIR.mkdir(parents=True, exist_ok=True)

start, end

## Ações e câmbio no Yahoo Finance

In [ ]:
equity_fx_tickers = [a["ticker"] for a in config["assets"] if a["category"] in ("equity", "fx")]
equities_fx = collect_equities_fx(equity_fx_tickers, start, end)  # dict {ticker: OHLCV}
pd.DataFrame({t: df["Close"] for t, df in equities_fx.items()}).tail()

## Futuros no Yahoo Finance

São coletados os preços brutos e, quando disponíveis, os fatores de rolagem.
A reconstrução usada no experimento principal é feita pelo script da base pública.

In [ ]:
futures_assets = {a["name"]: a["ticker"] for a in config["assets"] if a["category"] == "commodity"}

futures_raw = {}
futures_adjusted = {}
for name, ticker in futures_assets.items():
    raw_df = collect_futures_raw(ticker, start, end)  # OHLCV bruto
    futures_raw[ticker] = raw_df
    futures_adjusted[ticker] = adjust_ohlc_roll(raw_df)  # Sem fatores verificados: preserva os preços originais

futures_adjusted_close_df = pd.DataFrame({t: df["Close"] for t, df in futures_adjusted.items()})
futures_adjusted_close_df.tail()

## Indicadores físicos de soja e café do CEPEA

In [ ]:
cepea_soja = collect_cepea_spot("soja").loc[start:end]
cepea_cafe = collect_cepea_spot("cafe_arabica").loc[start:end]
cepea_soja.tail()

## Indicadores do BCB-SGS e do FRED

In [ ]:
macro_bcb = collect_bcb_sgs(start, end)
macro_bcb.tail()

In [ ]:
macro_fred = collect_fred(start, end)
macro_fred.tail()

## Volatilidade implícita

O VIX é coletado como referência do mercado externo. A variável `implied_vol_br`
vem de uma exportação manual do Bloomberg: IBOV Index, campo
`30DAY_IMPVOL_100.0%MNY_DF`, com cobertura de 19/06/2015 a 30/12/2024.
O arquivo usado pelo projeto é `data/raw/ibov_vol_bloomberg.xlsx`.
O IVol-BR do NEFIN, encerrado em abril de 2022, é apenas uma referência para
eventuais análises de robustez.

In [ ]:
implied_vol_us = collect_implied_vol_us(start, end)
implied_vol_br = load_bloomberg_implied_vol_br(OUT_DIR / "ibov_vol_bloomberg.xlsx")
implied_vol = pd.concat([implied_vol_us, implied_vol_br], axis=1, sort=False)
implied_vol.tail()

## Cobertura das séries e lacunas no período de 2005 a 2024

In [ ]:
checks = []
for ticker, df in equities_fx.items():
    checks.append(check_coverage(df["Close"], start, end, name=ticker))
for ticker, df in futures_adjusted.items():
    checks.append(check_coverage(df["Close"], start, end, name=f"{ticker}_yahoo_adjusted"))
checks.append(check_coverage(cepea_soja["price_brl"], start, end, name="cepea_soja"))
checks.append(check_coverage(cepea_cafe["price_brl"], start, end, name="cepea_cafe_arabica"))

MONTHLY_BCB_SERIES = {"ipca", "ibc_br"}  # selic e macro_fred sao diarias (freq default "B")
for col in macro_bcb.columns:
    freq = "MS" if col in MONTHLY_BCB_SERIES else "B"
    checks.append(check_coverage(macro_bcb[col], start, end, name=f"bcb_{col}", freq=freq))
for col in macro_fred.columns:
    checks.append(check_coverage(macro_fred[col], start, end, name=f"fred_{col}"))
checks.append(check_coverage(implied_vol["implied_vol_us"], start, end, name="implied_vol_us"))
checks.append(check_coverage(implied_vol["implied_vol_br"], start, end, name="implied_vol_br"))

summary = summarize_checks(checks)
summary

## Salvar em `data/raw/`

In [ ]:
pd.concat(equities_fx, axis=1, sort=False).to_parquet(OUT_DIR / "equities_fx.parquet")
pd.concat(futures_adjusted, axis=1, sort=False).to_parquet(OUT_DIR / "futures_adjusted_yahoo.parquet")
pd.concat(futures_raw, axis=1, sort=False).to_parquet(OUT_DIR / "futures_raw_yahoo.parquet")
cepea_soja.to_parquet(OUT_DIR / "cepea_soja.parquet")
cepea_cafe.to_parquet(OUT_DIR / "cepea_cafe_arabica.parquet")
macro_bcb.to_parquet(OUT_DIR / "macro_bcb.parquet")
macro_fred.to_parquet(OUT_DIR / "macro_fred.parquet")
implied_vol.to_parquet(OUT_DIR / "implied_vol.parquet")

tables_dir = PROJECT_ROOT / "reports" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(tables_dir / "data_coverage_summary.csv")

print("Coleta concluída — arquivos salvos em", OUT_DIR)